# RAG

In [3]:
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

openai_client = OpenAI()
documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, first install it from [https://ollama.com/download](https://ollama.com/download) for your operating system:\n\n- macOS: download and install the `.pkg`\n- Windows: download and install the `.msi`\n- Linux: run:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nThen open a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo check that the local server is running, you can also run:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf needed, you can restart the Ollama server with:\n\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'The FAQ context doesn’t mention **Olama** specifically.\n\nIf you mean running the course locally, the FAQ says you can do that instead of Codespaces if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. It also says to **document your setup and keep your environment reproducible**.\n\nIf you meant something else by “Olama,” let me know.'

In [7]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

## Asking without the tools

In [8]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—probably, but it depends on the course’s enrollment rules.\n\nIf you want, I can help you check:\n- whether it’s still open\n- any deadline or waitlist\n- prerequisites\n- how to enroll\n\nIf you already have the course name or link, send it to me and I’ll help you figure it out.'

## Defining the tool

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Sending the question with the tool

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join discovered the course"}', call_id='call_F0GtFln81SYzjgfpQsPIEig8', name='search', type='function_call', id='fc_0c9049a4bb3f8a27006a39186cb98481a2bceee3d98899db1e', namespace=None, status='completed')]

## Executing the function and sending the result back

In [17]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

## Asking the model again 

In [18]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf you want a certificate, make sure you submit your project while the course is still accepting submissions.'

## Token usage and cost of all these

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 35)

In [20]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Developer prompt

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## Function-call helper

In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment deadline join after course started FAQ"}
function_call: search {"query":"new student can I join the course now FAQ"}


## Full agent loop

In [25]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

- You can start learning anytime.
- You can also submit homework while the submission form is still open.
- If you want a certificate, you need to submit your project while the course is still accepting submissions.

If you want, I can also point you to the course starting materials or explain the certificate rules. Are there other areas you’d like to explore?


## Wrapping it in a function

In [26]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install Ollama locally run model start server"}
function_call: search {"query":"Ollama local setup run command localhost FAQ"}
function_call: search {"query":"run Ollama locally model pull serve FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s running, you should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(


'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s running, you should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restart the server with:\n```bash\nollama serve\n```\nOr in a notebook:

In [28]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join enrollment FAQ"}
function_call: search {"query":"late enrollment join course after start can I still join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted.

If you want, I can also help you figure out how to start catching up quickly or explain the certificate requirements in more detail.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out how to start catching up quickly or explain the certificate requirements in more detail.'

## Encouraging multiple searches

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course enrollment late can I join course discovered the course"}
iteration #2...
function_call: search {"query":"certificate join late discovered the course project still accepting submissions live cohort self-paced"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

One important caveat: if you want a certificate, you need to submit your project while submissions are still being accepted. Self-paced participation alone does not qualify for a certificate.

If you want, I can also help you with:
- how to start from the current point in the course
- whether homework is required for the certificate
- how the capstone/project works

Any other areas you want to explore?


'Yes — you can still join the course.\n\nOne important caveat: if you want a certificate, you need to submit your project while submissions are still being accepted. Self-paced participation alone does not qualify for a certificate.\n\nIf you want, I can also help you with:\n- how to start from the current point in the course\n- whether homework is required for the certificate\n- how the capstone/project works\n\nAny other areas you want to explore?'

## Restricting off topic questrions

In [30]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess definition opening"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. d4 d5  
2. c4

White offers the c4 pawn to try to **draw Black’s d-pawn away** from the center and gain **better control of the center**, especially the **e4** square.

A few quick points:
- It’s one of the oldest and most famous chess openings.
- Despite the name, it’s usually **not a real gambit** in the long term, because Black often can’t safely keep the pawn.
- It has two main forms:
  - **Queen’s Gambit Accepted**: Black takes the c4 pawn.
  - **Queen’s Gambit Declined**: Black doesn’t take it.

If you want, I can also explain **why it’s called a gambit** or show the **main lines** in simple terms.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c4 pawn to try to **draw Black’s d-pawn away** from the center and gain **better control of the center**, especially the **e4** square.\n\nA few quick points:\n- It’s one of the oldest and most famous chess openings.\n- Despite the name, it’s usually **not a real gambit** in the long term, because Black often can’t safely keep the pawn.\n- It has two main forms:\n  - **Queen’s Gambit Accepted**: Black takes the c4 pawn.\n  - **Queen’s Gambit Declined**: Black doesn’t take it.\n\nIf you want, I can also explain **why it’s called a gambit** or show the **main lines** in simple terms.'

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"gambit chess opening queen"}
iteration #3...
function_call: search {"query":"queen gambit chess opening"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so this looks off-topic for the course.

If you meant **Queen’s Gambit** in chess, I can’t answer that from the FAQ database here. If you want, I can help with another course-related question or topic.


'I couldn’t find a course FAQ entry for “queen gambit,” so this looks off-topic for the course.\n\nIf you meant **Queen’s Gambit** in chess, I can’t answer that from the FAQ database here. If you want, I can help with another course-related question or topic.'